<a href="https://colab.research.google.com/github/Ishalllll/Tokopedia-Product-Sentiment-Analysis/blob/main/Notebook/product_review_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Sastrawi
!pip install --quiet indoNLP Sastrawi scikit-learn nltk spacy

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import re
import string
import pandas as pd
from indoNLP.preprocessing import pipeline,replace_word_elongation,replace_slang

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


Model spaCy 'en_core_web_sm' berhasil dimuat.


# Muat Dataset

In [ ]:
df = pd.read_csv("review_product.csv")
print(f"Ukuran file \t: {df.shape}")
print(f"Nama Kolom \t: {df.columns}")
print(f"Jumlah Kolom Rating \t: {df['rating'].value_counts()}")
display(df.head())

Ukuran file 	: (40597, 11)
Nama Kolom 	: Index(['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0', 'text', 'rating',
       'category', 'product_name', 'product_id', 'sold', 'shop_id',
       'product_url'],
      dtype='object')
Jumlah Kolom Rating 	: rating
5    30305
4     7546
3     1825
1      542
2      379
Name: count, dtype: int64


,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,text,rating,category,product_name,product_id,sold,shop_id,product_url
0,0,0,1,Barang sesuai pesanan dan cepat sampai,5,pertukangan,Staples Dekorasi Staples Kayu + Refill 8mm - S...,418660637,1,1740837,https://www.tokopedia.com/shakaonline87/staple...
1,1,1,2,Barang bagus harga murah,5,pertukangan,STAPLE GUN ATS 3 WAY TACKER - STAPLES JOK TEMB...,416032545,11,1477109,https://www.tokopedia.com/juraganperkakas/stap...
2,2,2,3,Paket rapi...mantap....cepat....sampe ke tujuan,5,pertukangan,STAPLE GUN ATS 3 WAY TACKER - STAPLES JOK TEMB...,416032545,11,1477109,https://www.tokopedia.com/juraganperkakas/stap...
3,3,3,4,ya saya puas dgn barangnya,5,pertukangan,ALAT STAPLES TEMBAK &#40;AIR NAILER GUN&#41; O...,102279869,5,771395,https://www.tokopedia.com/kamarmesin/alat-stap...
4,4,4,5,Responya luar biasa b mantap,5,pertukangan,Isi Refill Staples Jok Kulit Motor / Staple Gu...,190679689,787,969999,https://www.tokopedia.com/mitrapersada/isi-ref...


# Pembersihan dan Pelabelan Data

In [ ]:
kolom_tak_perlu = [kolom for kolom in df.columns if kolom.startswith("Unnamed")]
df = df.drop(columns=kolom_tak_perlu)

def pelabelan(rating):
  if rating >= 4:
    return 1
  elif rating == 3:
    return 0
  else:
    return -1

df["Sentiment_label"] = df['rating'].apply(pelabelan)
display(df.head())

,text,rating,category,product_name,product_id,sold,shop_id,product_url,Sentiment_label,clean_text
0,Barang sesuai pesanan dan cepat sampai,5,pertukangan,Staples Dekorasi Staples Kayu + Refill 8mm - S...,418660637,1,1740837,https://www.tokopedia.com/shakaonline87/staple...,1,barang sesuai pesanan cepat
1,Barang bagus harga murah,5,pertukangan,STAPLE GUN ATS 3 WAY TACKER - STAPLES JOK TEMB...,416032545,11,1477109,https://www.tokopedia.com/juraganperkakas/stap...,1,barang bagus harga murah
2,Paket rapi...mantap....cepat....sampe ke tujuan,5,pertukangan,STAPLE GUN ATS 3 WAY TACKER - STAPLES JOK TEMB...,416032545,11,1477109,https://www.tokopedia.com/juraganperkakas/stap...,1,paket oke tujuan
3,ya saya puas dgn barangnya,5,pertukangan,ALAT STAPLES TEMBAK &#40;AIR NAILER GUN&#41; O...,102279869,5,771395,https://www.tokopedia.com/kamarmesin/alat-stap...,1,ya puas barangnya
4,Responya luar biasa b mantap,5,pertukangan,Isi Refill Staples Jok Kulit Motor / Staple Gu...,190679689,787,969999,https://www.tokopedia.com/mitrapersada/isi-ref...,1,respon b mantap


# Pre-Processing Data Set

In [ ]:
# labeling
ubah = {
  "aamin": "amin", "abalterimakasih": "terima kasih", "accept": "cepat", "admiin": "amin", "admin": "amin",
  "aksih": "terima kasih", "alamat": "lambat", "alamiin": "amin", "alhamdhulillah": "alhamdulillah",
  "alhamduillah": "alhamdulillah", "alhamduliah": "alhamdulillah", "alhamdulialh": "alhamdulillah",
  "alhamdulilah": "alhamdulillah", "alhamduliliiah": "alhamdulillah", "alhamdulillaah": "alhamdulillah",
  "alhamdulillh": "alhamdulillah", "alhamdullah": "alhamdulillah", "alhamdullilah": "alhamdulillah",
  "alhamdullillah": "alhamdulillah", "alhamfulilah": "alhamdulillah", "alhmdulilah": "alhamdulillah",
  "alhmdulillah": "alhamdulillah", "allhamdulilah": "alhamdulillah", "allhamdulilillah": "alhamdulillah",
  "allhamdulillah": "alhamdulillah", "amat": "sangat", "amien": "amin", "amiin": "amin", "arang": "barang",
  "atap": "mantap", "authentic": "ori", "baagggggus": "bagus", "baagguus": "bagus", "baagus": "bagus",
  "baarang": "barang", "baarangny": "barang", "baarng": "barang", "babgus": "bagus", "bad": "buruk",
  "baeang": "barang", "bafus": "bagus", "bagggggggus": "bagus", "baggus": "bagus", "bagis": "bagus", "bagius": "bagus",
  "bagoos": "bagus", "bagos": "bagus", "bags": "bagus", "bagu": "bagus", "bagua": "bagus", "bagud": "bagus",
  "baguis": "bagus", "bagusa": "bagus", "bagusas": "bagus", "bagusd": "bagus", "bagusds": "bagus", "bagussd": "bagus",
  "bagussz": "bagus", "baguus": "bagus", "baguusa": "bagus", "baguz": "bagus", "bagys": "bagus", "balang": "barang",
  "bang": "barang", "bangus": "bagus", "bara": "barang", "barabg": "barang", "barag": "barang", "baragn": "barang",
  "baramg": "barang", "baran": "barang", "baranb": "barang", "baranf": "barang", "baranga": "barang",
  "barangbga": "barang", "barange": "barang", "barangmya": "barang", "barangn": "barang", "barangna": "barang",
  "barangnga": "barang", "barangnnya": "barang", "barangny": "barang", "barangnys": "barang", "barangnyw": "barang",
  "barangx": "barang", "barangxa": "barang", "barangy": "barang", "barangya": "barang", "baranh": "barang",
  "baranng": "barang", "barano": "barang", "barans": "barang", "baranv": "barang", "bareng": "barang",
  "barg": "barang", "barnag": "barang", "barng": "barang", "barqng": "barang", "barrang": "barang",
  "barring": "barang", "barsng": "barang", "barus": "bagus", "batang": "barang", "baugus": "bagus", "baus": "bagus",
  "bawang": "barang", "bayang": "barang", "bbest": "terbaik", "berang": "barang", "best": "terbaik", "bguus": "bagus",
  "bogus": "bagus", "boks": "bos", "boos": "bos", "boots": "bos", "bosa": "bos", "bosq": "bos", "bozs": "bos",
  "brang": "barang", "brange": "barang", "brangx": "barang", "brangy": "barang", "brng": "barang", "bsgus": "bagus",
  "buaaggus": "bagus", "buagus": "bagus", "bubuk": "buruk", "bugus": "bagus", "buluk": "buruk", "buru": "buruk",
  "busuk": "buruk", "capat": "cepat", "ccepat": "cepat", "ceat": "cepat", "ceepet": "cepat", "ceoat": "cepat",
  "cepa": "cepat", "cepaat": "cepat", "cepag": "cepat", "cepan": "cepat", "cepar": "cepat", "cepatk": "cepat",
  "cepaty": "cepat", "cepay": "cepat", "cepeet": "cepat", "cepet": "cepat", "cepqt": "cepat", "cepst": "cepat",
  "cept": "cepat", "cheap": "murah", "cheat": "cepat", "cpat": "cepat", "cpeat": "cepat", "crpat": "cepat",
  "cuwepat": "cepat", "cwept": "cepat", "cwpat": "cepat", "danpengiriman": "pengiriman", "danrespon": "respon",
  "debat": "hebat", "dpengiriman": "pengiriman", "drespone": "respon","direspond": "respon", "engirimannya": "pengiriman",
  "excellent": "hebat", "fake": "palsu", "fasst": "cepat", "fast": "cepat", "fespon": "respon", "gambat": "lambat",
  "gaming": "amin", "garang": "barang", "golong": "tolong", "good": "bagus", "greaat": "hebat", "great": "hebat",
  "hagus": "bagus", "hambat": "lambat", "hemat": "hebat", "hoke": "oke", "imakasih": "terima kasih", "ip": "mantap",
  "jamin": "amin", "jarang": "barang", "jgpengiriman": "pengiriman", "jhos": "mantap", "joas": "mantap",
  "jobs": "mantap", "joos": "mantap", "josa": "mantap", "josd": "mantap", "jose": "mantap", "josh": "mantap",
  "josw": "mantap", "js": "mantap", "juos": "mantap", "kaasih": "terima kasih", "karang": "barang", "karen": "keren",
  "kasih": "terima kasih", "kasiih": "terima kasih", "ke": "oke", "keereen": "keren", "kemrn": "keren",
  "kepat": "cepat", "kere": "keren", "kerends": "keren", "kerens": "keren", "keuren": "keren", "kheren": "keren",
  "kmren": "keren", "kren": "keren", "ktolong": "tolong", "lama": "lambat", "lambaat": "lambat", "lamban": "lambat",
  "lebat": "hebat", "lmbat": "lambat", "lmbt": "lambat", "long": "tolong", "love": "suka", "maakasih": "terima kasih",
  "maanntuul": "mantap", "maantap": "mantap", "maantappoop": "mantap", "maantul": "mantap", "mabtab": "mantap",
  "maf": "maaf", "makadih": "terima kasih", "makas": "terima kasih", "makaseeh": "terima kasih",
  "makaseh": "terima kasih", "makash": "terima kasih", "makasiah": "terima kasih", "makasich": "terima kasih",
  "makasie": "terima kasih", "makasig": "terima kasih", "makasiih": "terima kasih", "makasiw": "terima kasih",
  "makasoh": "terima kasih", "makassi": "terima kasih", "makassih": "terima kasih", "makassiih": "terima kasih",
  "makkasih": "terima kasih", "maksh": "terima kasih", "maksi": "terima kasih", "maksiah": "terima kasih",
  "mamtaf": "maaf", "mamtap": "mantap", "mana": "mantap", "manager": "mager", "manatap": "mantap", "mancaaf": "maaf",
  "mancap": "mantap", "mancul": "mantap", "manghtabs": "mantap", "mangstab": "mantap", "mangstabs": "mantap",
  "mangtab": "mantap", "mangtabs": "mantap", "mangtap": "mantap", "manner": "mager", "manntaap": "mantap",
  "manntab": "mantap", "manntap": "mantap", "mannttap": "mantap", "manrap": "mantap", "manstab": "mantap",
  "manstabh": "mantap", "manstabz": "mantap", "manstap": "mantap", "manstaps": "mantap", "mant": "mantap",
  "manta": "mantap", "mantaab": "mantap", "mantaaf": "mantap", "mantaap": "mantap", "mantaaplah": "mantap",
  "mantaapps": "mantap", "mantaaps": "mantap", "mantaav": "mantap", "mantabbe": "mantap", "mantabbs": "mantap",
  "mantabe": "mantap", "mantabh": "mantap", "mantablah": "mantap", "mantabs": "mantap", "mantabz": "mantap",
  "mantaf": "mantap", "mantal": "mantap", "mantan": "mantap", "mantao": "mantap", "mantapak": "mantap",
  "mantapas": "mantap", "mantapbp": "mantap", "mantaph": "mantap", "mantapks": "mantap", "mantapl": "mantap",
  "mantaplah": "mantap", "mantaplp": "mantap", "mantaplpl": "mantap", "mantaplppl": "mantap", "mantapo": "mantap",
  "mantapoop": "mantap", "mantapop": "mantap", "mantappop": "mantap", "mantapps": "mantap", "mantappu": "mantap",
  "mantaps": "mantap", "mantapt": "mantap", "mantapz": "mantap", "mantapzs": "mantap", "mantapzxs": "mantap",
  "mantav": "mantap", "mantaz": "mantap", "manteb": "mantap", "manteeb": "mantap", "mantef": "mantap",
  "manten": "mantap", "manteplah": "mantap", "manteps": "mantap", "manthap": "mantap", "manthaps": "mantap",
  "mantiwab": "mantab", "mantol": "mantap", "mantp": "mantap", "mantqb": "mantab", "mantraap": "mantap",
  "mantrapz": "mantap", "manttaap": "mantap", "manttap": "mantap", "mantub": "mantap", "mantuul": "mantap",
  "manual": "mantap", "margotop": "banget", "markotob": "banget", "markototop": "banget", "marktop": "banget",
  "martkotop": "banget", "mata": "mantap", "matab": "mantab", "matabz": "mantab", "matap": "mantap",
  "matnap": "mantap", "merespon": "respon", "mkaasih": "terima kasih", "mkash": "terima kasih",
  "mkasi": "terima kasih", "mkasih": "terima kasih", "mksih": "terima kasih", "mkssih": "terima kasih",
  "mnatap": "mantap", "mntaap": "mantap", "mntab": "mantab", "mntap": "mantap", "mntb": "mantab", "mntep": "mantap",
  "mntp": "mantap", "moga": "semoga", "muaantap": "mantap", "muantab": "mantab", "muantabs": "mantab",
  "muantap": "mantap", "muantaph": "mantap", "muantapps": "mantap", "muantaps": "mantap", "muntap": "mantap",
  "mwantep": "mantap", "nantap": "mantap", "narang": "barang", "ngiriman": "pengiriman", "ngrespon": "respon",
  "nice": "bagus", "nntaap": "mantap", "ntap": "mantap", "oengiriman": "pengiriman", "oiike": "oke", "ok": "oke",
  "okay": "oke", "okeeh": "oke", "okeh": "oke", "okem": "oke", "okere": "keren", "okerre": "keren", "okey": "oke",
  "okhe": "oke", "okie": "oke", "okke": "oke", "okkew": "oke", "okre": "oke", "ook": "oke", "original": "ori",
  "os": "mantap", "pantul": "mantap", "parang": "barang", "pegiriman": "pengiriman", "pemgiriman": "pengiriman",
  "penfiriman": "pengiriman", "penggiriman": "pengiriman", "pengiiriman": "pengiriman", "pengirima": "pengiriman",
  "pengirimaan": "pengiriman", "pengirimaannya": "pengiriman", "pengirimah": "pengiriman", "pengirimam": "pengiriman",
  "pengirimana": "pengiriman", "pengirimanny": "pengiriman", "pengirimannye": "pengiriman",
  "pengirimanx": "pengiriman", "pengirimanya": "pengiriman", "pengirimaqn": "pengiriman", "pengirimin": "pengiriman",
  "pengiriminnya": "pengiriman", "pengirimn": "pengiriman", "pengirimnannya": "pengiriman",
  "pengirimnnya": "pengiriman", "pengirinan": "pengiriman", "pengiririman": "pengiriman", "pengirman": "pengiriman",
  "pengirmannya": "pengiriman", "pengiroman": "pengiriman", "pengiruman": "pengiriman", "pengorimannya": "pengiriman",
  "pengrimiman": "pengiriman", "pengrimn": "pengiriman", "penguriman": "pengiriman", "pengurimannya": "pengiriman",
  "penngiriman": "pengiriman", "perngiriman": "pengiriman", "pgiriman": "pengiriman", "pgriman": "pengiriman",
  "pingeriman": "pengiriman", "pingiriman": "pengiriman", "please": "tolong", "pls": "tolong",
  "pngiriman": "pengiriman", "pngirimannya": "pengiriman", "pngirimanya": "pengiriman", "pngirimn": "pengiriman",
  "pngirimnya": "pengiriman", "pngirimsn": "pengiriman", "pngirman": "pengiriman", "pngriman": "pengiriman",
  "pngrimn": "pengiriman", "prngiriman": "pengiriman", "pwmgiriman": "pengiriman", "pwngiriman": "pengiriman",
  "qoke": "oke", "rang": "barang", "reapon": "respon", "reaspon": "respon", "recomendasikan": "dianjurkan",
  "recommend": "dianjurkan", "recommended": "dianjurkan", "redpon": "respon",
  "rekomendasi": "dianjurkan", "rekspon": "respon", "repat": "cepat", "repon": "respon", "repons": "respon",
  "reposn": "respon", "repson": "respon", "resfon": "respon", "resond": "respon", "resp": "respon",
  "resphon": "respon", "respin": "respon", "respn": "respon", "respo": "respon", "respob": "respon",
  "respod": "respon", "respom": "respon", "responce": "respon", "respond": "respon", "responded": "respon",
  "responden": "respon", "responds": "respon", "respone": "respon", "respones": "respon", "responnse": "respon",
  "responny": "respon", "responnya": "respon", "respons": "respon", "response": "respon", "responses": "respon",
  "responsi": "cepat tanggap", "responsif": "cepat tanggap", "responsip": "cepat tanggap", "respont": "respon",
  "responx": "respon", "respony": "respon", "responya": "respon", "respoon": "respon", "respos": "respon",
  "respot": "respon", "respown": "respon", "respponnya": "respon", "respun": "respon", "rewpon": "respon",
  "rrespon": "respon", "rrspon": "respon", "rspon": "respon", "saip": "mantap", "scepat": "cepat", "sealer": "penjual",
  "sebarang": "barang", "seleer": "penjual", "seler": "penjual", "selera": "penjual", "selle": "penjual",
  "selleer": "penjual", "sellelr": "penjual", "sellers": "penjual", "semga": "semoga", "semiga": "semoga",
  "semmoga": "semoga", "semo": "semoga", "semoat": "semoga", "semog": "semoga", "semogs": "semoga", "semoha": "semoga",
  "semonga": "semoga", "shiip": "mantap", "si": "mantap", "siap": "mantap", "sicepat": "cepat", "siipb": "mantap",
  "siop": "mantap", "siph": "mantap", "sipmantaap": "mantap", "sipo": "mantap", "sips": "mantap", "sipz": "mantap",
  "skip": "mantap", "slip": "mantap", "slow": "lambat", "somoga": "semoga", "sorey": "sorry", "sory": "sorry",
  "spon": "respon", "ssi": "mantap", "ssiip": "mantap", "ssip": "mantap", "ssuip": "mantap", "sua": "suka",
  "suip": "mantap", "sukaak": "suka", "sukak": "suka", "sukka": "suka", "super": "hebat", "superr": "hebat",
  "suuka": "suka", "syiip": "mantap", "syuka": "suka", "syukak": "suka", "syukka": "suka", "syuuka": "suka",
  "tarimakasi": "terima kasih", "teimaksih": "terima kasih", "teirmaksih": "terima kasih", "tepaat": "cepat",
  "tepat": "cepat", "terimaka": "terima kasih", "terimakaaih": "terima kasih", "terimakadih": "terima kasih",
  "terimakaih": "terima kasih", "terimakash": "terima kasih", "terimakasi": "terima kasih",
  "terimakasiah": "terima kasih", "terimakasih": "terima kasih", "terimakasiih": "terima kasih",
  "terimakasij": "terima kasih", "terimakassih": "terima kasih", "terimaksi": "terima kasih",
  "terimakskh": "terima kasih", "terimamakasih": "terima kasih", "terimankasih": "terima kasih",
  "terimasih": "terima kasih", "terimkasih": "terima kasih", "terimksh": "terima kasih", "terimksih": "terima kasih",
  "terimmasih": "terima kasih", "termakasih": "terima kasih", "termiakasih": "terima kasih",
  "terumakasih": "terima kasih", "tespon": "respon", "tetimakasih": "terima kasih", "thank": "terima kasih",
  "thankks": "terima kasih", "thanks": "terima kasih", "thannk": "terima kasih", "thannkks": "terima kasih",
  "thx": "terima kasih", "tlambat": "lambat", "tlng": "tolong", "tlong": "tolong", "tmakasih": "terima kasih",
  "toop": "mantap", "top": "mantap", "topmarkotop": "banget", "tops": "top", "tq": "terima kasih",
  "trakasih": "terima kasih", "trimakasaih": "terima kasih", "trimakasi": "terima kasih", "trimakasih": "terima kasih",
  "trimakasiih": "terima kasih", "trimaksh": "terima kasih", "trimaksi": "terima kasih", "trimaksih": "terima kasih",
  "trimkasi": "terima kasih", "trimkasih": "terima kasih", "trimksih": "terima kasih", "trlambat": "lambat",
  "trmakash": "terima kasih", "trmakasih": "terima kasih", "trmkasih": "terima kasih", "trrimakasih": "terima kasih",
  "tterimakasih": "terima kasih", "ttop": "mantap", "vagus": "bagus", "varang": "barang", "vepat": "cepat",
  "verry": "sangat", "very": "sangat", "woke": "oke", "yerimakasih": "terima kasih", "sip":"mantap", "tks" : "terima kasih",
  "mantul":"mantap betul","ok":"oke"," n ":" dan ","recomended":"dianjurkan", "thankyou":"terima kasih", "gx":"tidak", "sdh":"sudah","jg":"juga"
}

stop_word_penting = {
  "tidak", "tak", "enggak", "belum",
  "jangan", "harus", "sangat", "kurang",
  "lebih", "amat", "cukup", "sekali",
  "baru", "baik", "paling","terlalu"
}

In [ ]:
pattern = re.compile(r'\b(' + '|'.join(k for k in ubah.keys()) + r')\b')
def ubah_bentuk(kata):
  kata_ = kata.group(0)
  return ubah.get(kata_, kata_)

In [ ]:
regex = re.compile(r"([^g])\1{2,}")
def hapus_pengulangan_huruf(kata,repetisi=1):
  return regex.sub(lambda x: x.group(1) * repetisi, kata)

In [ ]:
daftar_stopword = set(stopwords.words("indonesian"))-stop_word_penting

def preprocessing(teks):
  lowercase = teks.lower()
  pengulangan1 = hapus_pengulangan_huruf(lowercase)
  pengulangan2 = replace_word_elongation(pengulangan1)
  ubah_kata1 = replace_slang(pengulangan2)
  ubah_kata2 = pattern.sub(ubah_bentuk,ubah_kata1)
  tokenisasi = nltk.sent_tokenize(ubah_kata2)
  semua_kata = []
  for word in tokenisasi:
    semua_kata.extend(word_tokenize(word))
  teks_ = [
      kata for kata in semua_kata
      if kata not in string.punctuation
      and kata.isalnum()
      and kata not in daftar_stopword
  ]
  return " ".join(teks_)

df["clean_text"] = df["text"].astype(str).apply(preprocessing)
display(df.head())
df.to_csv("data bersihhhhhh.csv")

,text,rating,category,product_name,product_id,sold,shop_id,product_url,Sentiment_label,clean_text
0,Barang sesuai pesanan dan cepat sampai,5,pertukangan,Staples Dekorasi Staples Kayu + Refill 8mm - S...,418660637,1,1740837,https://www.tokopedia.com/shakaonline87/staple...,1,barang sesuai pesanan cepat
1,Barang bagus harga murah,5,pertukangan,STAPLE GUN ATS 3 WAY TACKER - STAPLES JOK TEMB...,416032545,11,1477109,https://www.tokopedia.com/juraganperkakas/stap...,1,barang bagus harga murah
2,Paket rapi...mantap....cepat....sampe ke tujuan,5,pertukangan,STAPLE GUN ATS 3 WAY TACKER - STAPLES JOK TEMB...,416032545,11,1477109,https://www.tokopedia.com/juraganperkakas/stap...,1,paket oke tujuan
3,ya saya puas dgn barangnya,5,pertukangan,ALAT STAPLES TEMBAK &#40;AIR NAILER GUN&#41; O...,102279869,5,771395,https://www.tokopedia.com/kamarmesin/alat-stap...,1,ya puas barangnya
4,Responya luar biasa b mantap,5,pertukangan,Isi Refill Staples Jok Kulit Motor / Staple Gu...,190679689,787,969999,https://www.tokopedia.com/mitrapersada/isi-ref...,1,respon b mantap


# Analisis Sentimen

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

tfidf_vectorizer = TfidfVectorizer()
tfidf_features = tfidf_vectorizer.fit_transform(df["clean_text"])

print("ukuran dari vectorizer: ", tfidf_features.shape)

with open("vectorizer_Muhammad_Faishal_Ardiansyahr.pkl","wb") as f:
  pickle.dump(tfidf_vectorizer, f)

ukuran dari vectorizer:  (40597, 10805)


In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

x = tfidf_features
y = np.array(df["Sentiment_label"])

In [ ]:
print(df["Sentiment_label"].value_counts())
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, stratify=y, random_state=42)

Sentiment_label
 1    37851
 0     1825
-1      921
Name: count, dtype: int64


In [ ]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.naive_bayes import ComplementNB

# (a) Oversample minor tanpa threshold
ros = RandomOverSampler(sampling_strategy={-1:1500,0:1500}, random_state=42)
x_train_bal, y_train_bal = ros.fit_resample(x_train, y_train)

# (b) Fit ComplementNB
model = ComplementNB(alpha=0.1)
model.fit(x_train_bal, y_train_bal)

with open("model_Muhammad_Faishal_Ardiansyah.pkl","wb") as f:
  pickle.dump(model,f)

# Predict

In [ ]:
with open("model_Muhammad_Faishal_Ardiansyah.pkl","rb") as m_file:
  model_ = pickle.load(m_file)

with open("vectorizer_Muhammad_Faishal_Ardiansyahr.pkl","rb") as v_file:
  vectorizer_ = pickle.load(v_file)

In [ ]:
testing = pd.read_csv("question.csv")
processed_text_ = [preprocessing(kalimat) for kalimat in testing["text"]]
text_vector = vectorizer_.transform(processed_text_)


testing["result"] = model_.predict(text_vector)
label_map = {1: "positif", 0: "netral", -1: "negatif"}
testing["result"] = testing["result"].map(label_map)

display(testing)

,Unnamed: 0,text,category,product_name,product_id,result
0,0,Warnanya tdk sesuai pesanan. Tdk ada chat/pemb...,elektronik,USB HUB 3.0 7 port by DIGIGEAR HIGH SPEED 1.2 ...,170197447,negatif
1,1,Thanks gan barang bagus.smoga awet,handphone,Holder anti hujan &amp; copet untuk smartphone...,37564148,positif
2,2,barang sesuai deskripsi.. laptop dapat di-charge,elektronik,Adaptor Charger Laptop Toshiba Satellite C800 ...,254339251,positif
3,3,Goood product.......... ............,handphone,Silikon Case Blackberry Bold 9000 Hitam Gratis...,4837401,positif
4,4,Bagus pengiriman cepat.,handphone,Audio Splitter Jack 3.5mm to dual female U Sha...,270447288,positif
5,5,Terima kasih pesanannya sdh sampai,fashion,Celana jeans anak pendek,294837423,positif
6,6,"barangnya cacat, apa tidak diperiksa dulu?",handphone,Tempelan / Perekat Gurita Hp Buat / untuk Goje...,56486193,negatif
7,7,Barang sesuai pesanan,elektronik,Baterai Acer Aspire One 722 D255 D257 D260 D27...,189413883,positif
8,8,"Udah order ternyata stok ga tersedia, harusny...",fashion,Sepatu Bata Dairy Black - 6516353,194223236,netral
9,9,Pengiriman barang lambat,elektronik,USB HUB 4 port USB 3.0 / USB HUB 3.0 &#40;4por...,182937734,netral


In [ ]:
# evaluasi
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(x_test)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=4))

Confusion Matrix:
[[ 124   45   15]
 [ 132   68  165]
 [ 629  647 6295]]
Classification Report:
              precision    recall  f1-score   support

          -1     0.1401    0.6739    0.2320       184
           0     0.0895    0.1863    0.1209       365
           1     0.9722    0.8315    0.8963      7571

    accuracy                         0.7989      8120
   macro avg     0.4006    0.5639    0.4164      8120
weighted avg     0.9137    0.7989    0.8464      8120

